<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/Experiment_5_Fine_Tuning_using_Parameter_Efficient_Techniques_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages (run once)
!pip install torch transformers datasets peft evaluate accelerate huggingface_hub --quiet
!pip install --upgrade torchao --quiet
!pip install --upgrade transformers --quiet

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

W1110 16:37:09.074000 399 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 16:37:09.115000 399 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Using device: cuda


In [3]:
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 2  # e.g. positive / negative for a sentiment domain dataset

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load a domain-specific dataset (replace with your own dataset/path)
dataset = load_dataset("stanfordnlp/imdb")

# Use a small subset for quick experimentation; use full data in practice
train_ds = dataset["train"].shuffle(seed=42).select(range(2000))
eval_ds = dataset["test"].shuffle(seed=42).select(range(500))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length")

In [4]:
train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load the pretrained base model with a classification head
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

print("Data preprocessed and model loaded successfully!")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Data preprocessed and model loaded successfully!


In [5]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,            # sequence classification task
    r=8,                                   # rank of the low-rank decomposition
    lora_alpha=16,                         # scaling factor applied to LoRA updates
    lora_dropout=0.1,                      # dropout applied inside LoRA layers
    target_modules=["q_lin", "v_lin"]      # attention query/value layers (DistilBERT naming)
)

# Wrap the base model: original weights are frozen, LoRA adapters are added
peft_model = get_peft_model(base_model, lora_config).to(device)

# Show how few parameters are actually trainable compared to the total
peft_model.print_trainable_parameters()
# Typical output: "trainable params: ~0.7M || all params: ~67M || trainable%: ~1.1"

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [6]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    accuracy = evaluate.load("accuracy")
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# Training settings
training_args = TrainingArguments(
    output_dir="./lora-distilbert-imdb",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
)

# Create the trainer (this was missing)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.417478,0.308231,0.868000
2,0.266588,0.308669,0.870000
3,0.252895,0.281027,0.882000


TrainOutput(global_step=375, training_loss=0.33387390518188476, metrics={'train_runtime': 271.4756, 'train_samples_per_second': 22.101, 'train_steps_per_second': 1.381, 'total_flos': 808436441088000.0, 'train_loss': 0.33387390518188476, 'epoch': 3.0})

In [7]:
# Evaluate the fine-tuned (LoRA) model on the held-out evaluation set
lora_results = trainer.evaluate()
print("LoRA fine-tuned model results:", lora_results)
# Example: {'eval_loss': 0.28, 'eval_accuracy': 0.91, ...}

Training Loss,Validation Loss,Epoch,Accuracy
0.252895,0.281027,3,0.882000


LoRA fine-tuned model results: {'eval_loss': 0.2810274660587311, 'eval_accuracy': 0.882}


In [8]:
# Build a fresh baseline model: pretrained encoder + untrained head,
# with NO fine-tuning / NO LoRA adapters applied
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

baseline_trainer = Trainer(
    model=baseline_model,
    args=training_args,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

baseline_results = baseline_trainer.evaluate()
print("Baseline (no fine-tuning) results:", baseline_results)

# Side-by-side comparison
print("\n--- Comparison ---")
print(f"Baseline accuracy       : {baseline_results['eval_accuracy']:.4f}")
print(f"LoRA fine-tuned accuracy: {lora_results['eval_accuracy']:.4f}")

improvement = (lora_results['eval_accuracy'] - baseline_results['eval_accuracy']) * 100
print(f"Improvement             : {improvement:.2f} percentage points")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.694384,0,0.508000


Baseline (no fine-tuning) results: {'eval_loss': 0.6943844556808472, 'eval_accuracy': 0.508}

--- Comparison ---
Baseline accuracy       : 0.5080
LoRA fine-tuned accuracy: 0.8820
Improvement             : 37.40 percentage points
